# MicroLive Pipeline Benchmark

This notebook benchmarks the performance of the MicroLive analysis pipeline, measuring the execution time of each major step:
- Data loading
- Photobleaching correction
- Cell segmentation
- Particle tracking
- MSD analysis

## Generating Simulation Data

Before running this benchmark, generate the simulation data:

```bash
cd simulations
python run_simulation.py --config config_simple.yaml --output results_single_cell
```

In [1]:
# MicroLive imports
from microlive import microscopy as mi
from microlive.utils.device import check_gpu_status

# Standard scientific imports
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path
import time
import platform
import psutil

# System Information
print("System Information:")
print(f"OS: {platform.system()} {platform.release()}")
print(f"CPU: {platform.processor()}")
print(f"Python: {platform.python_version()}")
print(f"RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")

# GPU Status
check_gpu_status()

# Benchmark storage
benchmark_results = {}

def log_time(step_name, start_time):
    duration = time.time() - start_time
    benchmark_results[step_name] = duration
    print(f"[{step_name}] Execution time: {duration:.4f} seconds")
    return duration

System Information:
OS: Darwin 25.2.0
CPU: arm
Python: 3.10.19
RAM: 36.00 GB
PyTorch version: 2.10.0.dev20251205
✅ MPS available: Apple Silicon GPU (MPS)


## 1. Data Loading and Preparation

In [2]:
step_name = "Data Loading"
start_time = time.time()

# Parameters from simulation config
pixel_size_xy_nm = 130
voxel_size_z_nm = 300
time_interval_sec = 5.0
list_voxels = [voxel_size_z_nm, pixel_size_xy_nm]

# File Path - using single cell simulation
file_path = Path("../simulations/results_single_cell/simulated_spots.tif")

if not file_path.exists():
    raise FileNotFoundError(f"Run simulation first: cd simulations && python run_simulation.py --config config_simple.yaml --output results_single_cell")

print(f"Loading data from: {file_path}")
image_data = tifffile.imread(file_path)
print(f"Original shape (TCZYX): {image_data.shape}")

# Convert TCZYX -> TZYXC for MicroLive
image = np.transpose(image_data, (0, 2, 3, 4, 1)).astype(np.uint16)
print(f"Converted shape (TZYXC): {image.shape}")

log_time(step_name, start_time)

Loading data from: ../simulations/results_single_cell/simulated_spots.tif
Original shape (TCZYX): (120, 3, 10, 512, 512)
Converted shape (TZYXC): (120, 10, 512, 512, 3)
[Data Loading] Execution time: 1.6024 seconds


1.6023879051208496

## 2. Photobleaching Correction

In [3]:
step_name = "Photobleaching Correction"
start_time = time.time()

# Create mask from first frame
ref_image = image[0, :, :, :, 0].max(axis=0)
mask_YX = (ref_image > np.mean(ref_image)).astype(np.uint8)

corrector = mi.Photobleaching(
    image_TZYXC=image,
    mask_YX=mask_YX,
    show_plot=False,
    time_interval_seconds=time_interval_sec,
    mode="inside_cell"
)

image_corrected, _ = corrector.apply_photobleaching_correction()
image_corrected = image_corrected.astype(np.uint16)

log_time(step_name, start_time)
print(f"Corrected image shape: {image_corrected.shape}")

[Photobleaching Correction] Execution time: 1.2222 seconds
Corrected image shape: (120, 10, 512, 512, 3)


## 3. Cell Segmentation

In [4]:
step_name = "Cell Segmentation"
start_time = time.time()

segmentation_input = image_corrected[0, :, :, :, 0].max(axis=0)

segmentator = mi.CellSegmentationWatershed(
    image=segmentation_input,
    expected_radius=80,
    min_object_size=500,
    threshold_method="li",
    separation_size=1
)

mask_cytosol = segmentator.apply_watershed()

log_time(step_name, start_time)
print(f"Segmentation complete. Mask shape: {mask_cytosol.shape}")

[Cell Segmentation] Execution time: 0.0753 seconds
Segmentation complete. Mask shape: (512, 512)


## 4. Particle Tracking

In [5]:
step_name = "Particle Tracking"
start_time = time.time()

tracking_params = {
    "channels_spots": [0],
    "channels_cytosol": [0],
    "channels_nucleus": [None],
    "threshold_for_spot_detection": 2000,
    "yx_spot_size_in_px": 5,
    "z_spot_size_in_px": 2,
    "cluster_radius_nm": 500,
    "min_length_trajectory": 20,
    "maximum_range_search_pixels": 7,
    "memory": 0,
    "link_using_3d_coordinates": False,
    "use_maximum_projection": True,
}

binary_mask = (mask_cytosol > 0).astype(bool)

tracker = mi.ParticleTracking(
    image=image_corrected,
    list_voxels=list_voxels,
    masks=binary_mask,
    step_size_in_sec=time_interval_sec,
    **tracking_params
)

list_dataframes, _ = tracker.run()
df_tracking = list_dataframes[0] if list_dataframes else pd.DataFrame()

log_time(step_name, start_time)

# Report trajectories (unique particles)
if not df_tracking.empty and "particle" in df_tracking.columns:
    n_trajectories = df_tracking["particle"].nunique()
    n_observations = len(df_tracking)
    print(f"Tracking complete.")
    print(f"  Spot observations: {n_observations}")
    print(f"  Unique trajectories: {n_trajectories}")
else:
    print("No spots detected.")

[Particle Tracking] Execution time: 28.0880 seconds
Tracking complete.
  Spot observations: 1442
  Unique trajectories: 24


## 5. MSD Analysis

In [6]:
step_name = "MSD Analysis"
start_time = time.time()

if not df_tracking.empty and "particle" in df_tracking.columns:
    microns_per_pixel = pixel_size_xy_nm / 1000
    min_length = 20
    traj_lengths = df_tracking["particle"].value_counts()
    valid_particles = traj_lengths[traj_lengths >= min_length].index
    df_filtered = df_tracking[df_tracking["particle"].isin(valid_particles)].copy()

    if not df_filtered.empty:
        msd_analyzer = mi.ParticleMotion(
            trackpy_dataframe=df_filtered,
            microns_per_pixel=microns_per_pixel,
            step_size_in_sec=time_interval_sec,
            max_lagtime=30,
            show_plot=False,
            is_3d=False,
            max_fit_points=20
        )
        diffusion_coeff, _, _, _, _, _, _ = msd_analyzer.calculate_msd()
        print(f"MSD Analysis complete. D = {diffusion_coeff:.4f} µm²/s")
    else:
        print("No valid trajectories for MSD analysis.")
else:
    print("No tracking data available.")

log_time(step_name, start_time)

MSD Analysis complete. D = 0.0002 µm²/s
[MSD Analysis] Execution time: 0.0371 seconds


0.03713202476501465

## Performance Summary

In [7]:
print("\n--- MicroLive Pipeline Performance Summary ---")
total_time = 0
for step, duration in benchmark_results.items():
    print(f"{step}: {duration:.4f} seconds")
    total_time += duration

print(f"\nTotal Execution Time: {total_time:.4f} seconds")

if image is not None:
    data_size_mb = image.nbytes / (1024 * 1024)
    num_frames = image.shape[0]
    print(f"Processed Data Size: {data_size_mb:.2f} MB")
    print(f"Total Frames: {num_frames}")
    print(f"Throughput: {data_size_mb / total_time:.2f} MB/s")


--- MicroLive Pipeline Performance Summary ---
Data Loading: 1.6024 seconds
Photobleaching Correction: 1.2222 seconds
Cell Segmentation: 0.0753 seconds
Particle Tracking: 28.0880 seconds
MSD Analysis: 0.0371 seconds

Total Execution Time: 31.0250 seconds
Processed Data Size: 1800.00 MB
Total Frames: 120
Throughput: 58.02 MB/s
